### 设置agent名字以及系统提示词

In [3]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(override=True)

True

In [6]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek:deepseek-v4-flash"
)

agent = create_agent(
    model=model,
    name="agnet01",   # 设置名称
    # 设置系统提示词
    system_prompt="""
        你是数学家


        输出格式:
            原题目：
            计算过程：
            答案：
    """
)

messages = {
    "messages":[
        SystemMessage(content="你是一个精通数学的数学家，熟练的讲解数学问题"),
        HumanMessage(content="100 + 20 * 3 = ?")
    ]
}

response = agent.invoke(messages)
rprint(response)

{
    'messages': [
        SystemMessage(
            content='你是一个精通数学的数学家，熟练的讲解数学问题',
            additional_kwargs={},
            response_metadata={},
            id='31a5ea03-5698-4057-b8d1-9c1c9f5cd753'
        ),
        HumanMessage(
            content='100 + 20 * 3 = ?',
            additional_kwargs={},
            response_metadata={},
            id='c1570879-4c91-467f-ba2c-cff8e1e5de2a'
        ),
        AIMessage(
            content='原题目：100 + 20 * 3 = ?\n\n计算过程：  \n先算乘法：20 × 3 = 60  \n再算加法：100 + 60 = 
160\n\n答案：160',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '我们需要回答中文。用户说“你是数学家 输出格式: 原题目： 计算过程： 答案： 
你是一个精通数学的数学家，熟练的讲解数学问题”。题目 100 + 20 * 3 = ? 需要按格式。需要注意运算优先级。计算 
20*3=60，加100=160。给出简短过程。确保格式。可能不要多余。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 126,
                    'prompt_tokens': 72,
                    'total_tokens': 198,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 82,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 72
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '7c90782d-5f10-48a2-a1a0-72e2d76a9ae8',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='agnet01',
            id='lc_run--01a12227-8abc-7681-aa03-127ae9ee3898-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 72,
                'output_tokens': 126,
                'total_tokens': 198,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 82}
            }
        )
    ]
}

### 2 结构化输出

ProviderStrategy

In [13]:
model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={
        "thinking": {"type": "disabled"}
    },
      model_kwargs={
        "response_format": {"type": "json_object"}
    },
)

# 使用pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name:str = Field(description="用户姓名")
    email:str = Field(description="用户邮箱")
    phone:str = Field(description="用户电话")

agent = create_agent(
    model=model,
    name="agnet01",   # 设置名称
    system_prompt="""
        你是信息提取助手。
        输出结果必须是 json，只返回json字符串，不要任何多余文字、解释、markdown。
    """,
)

response = agent.invoke({
    "messages":[
        HumanMessage(content="一下信息中提取用户信息，小明邮箱是spf20031014@163.com,电话是18242141179")
    ]
})

c = ContractInfo.model_validate_json(response['messages'][-1].content)

rprint(c)

ContractInfo(name='小明', email='spf20031014@163.com', phone='18242141179')

### 2.1 结构化输出2

适用于不支持原生结构化输出的模型，使用ToolStrategy

In [5]:
model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

# 使用pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name:str = Field(description="用户姓名")
    email:str = Field(description="用户邮箱")
    phone:str = Field(description="用户电话")

agent = create_agent(
    model=model,
    name="agnet01",   # 设置名称
    system_prompt="""
        你是信息提取助手。
        输出结果必须是 json，只返回json字符串，不要任何多余文字、解释、markdown。
    """,
    response_format=ToolStrategy(ContractInfo)
)

response = agent.invoke({
    "messages":[
        HumanMessage(content="一下信息中提取用户信息，小明邮箱是spf20031014@163.com,电话是18242141179")
    ]
})

# c = ContractInfo.model_validate_json(response['messages'][-1].content)

rprint(response["structured_response"])

ContractInfo(name='小明', email='spf20031014@163.com', phone='18242141179')